In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,zero,5,0,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,zero,5,0,0.0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,2,zero,5,0,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,zero,5,0,0.0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,3,zero,5,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,other_causes,other_causes,95_plus,severe,3,zero,52,0,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,zero,52,0,0.0
539997,ylls,cause,other_causes,other_causes,95_plus,severe,4,zero,52,0,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,52,0,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  591306.980884
                                  2                  469119.806421
                                  3                  412121.233930
                                  4                  376933.536670
                                  5                  347363.884977
intervention  maternal_disorders  1                  591306.980884
                                  2                  469119.806421
                                  3                  412121.233930
                                  4                  376933.536670
                                  5                  347363.884977
zero          maternal_disorders  1                  603434.997467
                                  2                  476918.138162
                                  3                  419228.501520
                                  4                  383307.358238
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,1,zero,5,0,0.834648
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,zero,5,0,0.000000
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,zero,5,0,0.000000
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,zero,5,0,0.000000
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,zero,5,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,parturition,95_plus,severe,5,zero,52,0,0.000000
1889996,ylds,cause,pregnancy,postpartum,95_plus,severe,5,zero,52,0,0.000000
1889997,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,52,0,0.000000
1889998,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,zero,52,0,0.000000


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
                                  5                   45659.951633
              maternal_disorders  1                     164.936997
                                  2                     114.588556
                                  3                      92.849802
                                  4                      96.982363
                                  5                      98.453235
intervention  anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
            

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
                                  5                   45659.951633
              maternal_disorders  1                  591471.917880
                                  2                  469234.394977
                                  3                  412214.083731
                                  4                  377030.519033
                                  5                  347462.338212
intervention  anemia              1                  117697.275511
                                  2                   93116.577964
                                  3                   75010.563497
                                  4                   62893.286069
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,intervention,97,0,0.000000
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,intervention,97,0,0.000000
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,intervention,97,0,0.000000
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,intervention,97,0,0.000000
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,intervention,97,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
47995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,93,0,3762.838756
47996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,93,0,3509.740317
47997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,93,0,3779.499962
47998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,93,0,3782.549571


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.858785e+07
                      2                  1.536768e+07
                      3                  1.374935e+07
                      4                  1.289434e+07
                      5                  1.233837e+07
intervention  lbwsg   1                  1.858785e+07
                      2                  1.536768e+07
                      3                  1.374935e+07
                      4                  1.289434e+07
                      5                  1.233837e+07
zero          lbwsg   1                  1.864088e+07
                      2                  1.540228e+07
                      3                  1.377754e+07
                      4                  1.291749e+07
                      5                  1.235066e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1880.299813,zero
1,Female,0.0,0.019178,2,1491.543371,zero
2,Female,0.0,0.019178,3,1302.737481,zero
3,Female,0.0,0.019178,4,1106.922745,zero
4,Female,0.0,0.019178,5,791.187078,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,827.888249,intervention
746,Male,95.0,125.000000,2,844.738857,intervention
747,Male,95.0,125.000000,3,868.381929,intervention
748,Male,95.0,125.000000,4,894.402190,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.697593e+06
                      2                  1.647863e+06
                      3                  1.670509e+06
                      4                  1.576334e+06
                      5                  1.394792e+06
intervention  anemia  1                  1.697593e+06
                      2                  1.647863e+06
                      3                  1.670509e+06
                      4                  1.576334e+06
                      5                  1.394792e+06
zero          anemia  1                  1.832088e+06
                      2                  1.779191e+06
                      3                  1.788230e+06
                      4                  1.676075e+06
                      5                  1.446281e+06
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-574287.5006291382

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  631678.673534
                      2                  510158.641426
                      3                  465380.058107
                      4                  393300.671783
                      5                  298560.903858
intervention  anemia  1                  631678.673534
                      2                  510158.641426
                      3                  465380.058107
                      4                  393300.671783
                      5                  298560.903858
zero          anemia  1                  671739.650416
                      2                  543859.134981
                      3                  493148.121281
                      4                  414117.563193
                      5                  307876.311574
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-131661.83273744863

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.117924e+06
                      2                  3.702880e+06
                      3                  3.629766e+06
                      4                  3.328091e+06
                      5                  2.956441e+06
intervention  anemia  1                  4.117924e+06
                      2                  3.702880e+06
                      3                  3.629766e+06
                      4                  3.328091e+06
                      5                  2.956441e+06
zero          anemia  1                  4.464303e+06
                      2                  4.009704e+06
                      3                  3.894623e+06
                      4                  3.542319e+06
                      5                  3.062869e+06
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  397090.296531
                      2                  346921.324136
                      3                  308402.467752
                      4                  288789.499819
                      5                  239828.039930
baseline      ntd     1                  368777.931735
                      2                  325776.815913
                      3                  292471.014886
                      4                  275785.753509
                      5                  234875.149488
intervention  ntd     1                  209395.616553
                      2                  199046.270081
                      3                  191327.190047
                      4                  189716.676021
                      5                  196846.917043
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.235621e+06
                                  2                  3.795997e+06
                                  3                  3.704777e+06
                                  4                  3.390984e+06
                                  5                  3.002101e+06
              lbwsg               1                  1.858785e+07
                                  2                  1.536768e+07
                                  3                  1.374935e+07
                                  4                  1.289434e+07
                                  5                  1.233837e+07
              maternal_disorders  1                  5.914719e+05
                                  2                  4.692344e+05
                                  3                  4.122141e+05
                                  4                  3.770305e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)